# Hiệu năng pipeline

Wall time và các đỉnh tài nguyên phải được đọc cùng nhãn run, input và môi trường. Không suy hiệu năng arXiv từ fixtures.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
sys.path.insert(0, str(ROOT / 'utils'))
from report_utils import load_run
run_name, summary, benchmark, samples = load_run(ROOT)
display(Markdown(f"**Run: {run_name} — label: {benchmark['label']}**"))
if benchmark['label'] == 'synthetic_demo':
    display(Markdown('> **DEMO OFFLINE: các con số không phải kết quả corpus thật hoặc benchmark Colab.**'))
elif benchmark['label'] == 'colab_cpu_sample':
    display(Markdown('> **MẪU COLAB CPU-ONLY: một paper được giao, không phải toàn bộ assignment 25.000 ID.**'))
else:
    display(Markdown('> Lượt gọi API thật; cần đối chiếu môi trường Colab CPU-only, phạm vi input và logs trước nghiệm thu.'))


In [ ]:
display(pd.DataFrame([{'wall_seconds': benchmark['wall_seconds'], 'pipeline_exit': benchmark['exit_code'], 'discovery_seconds': benchmark['timings']['discovery_total_seconds'], 'mean_paper_seconds': benchmark['timings']['mean_paper_seconds'], 'cached_events': benchmark['timings']['cached_events'], 'sample_interval_seconds': benchmark['sample_interval_seconds'], 'access_errors': benchmark['measurement_access_errors']}]))
env = {k:v for k,v in benchmark['environment'].items() if k != 'packages'}
display(pd.DataFrame([{'field': k, 'value': str(v)} for k,v in env.items()]))
display(pd.DataFrame([{'package': k,'version':v} for k,v in benchmark['environment'].get('packages',{}).items()]))

## Thời gian từng bước và từng paper

Paper lỗi cũng được tính trong attempted-input timing; alias cùng paper được cộng. Tổng stage không gồm mọi overhead của wall time. Events đang dở khi ngắt chưa có full duration.

In [ ]:
stage = benchmark['timings']['stage_seconds']
if stage:
    plt.figure(figsize=(9,4)); plt.bar(stage.keys(),stage.values()); plt.ylabel('Total seconds / stage'); plt.xticks(rotation=25); plt.tight_layout(); plt.show()
papers = benchmark['timings']['per_paper_seconds']
display(pd.DataFrame([{'paper_id': k, 'seconds': v} for k,v in papers.items()]))
if papers:
    plt.figure(figsize=(8,4)); plt.hist(list(papers.values()),bins=min(20,len(papers))); plt.xlabel('Seconds / paper'); plt.ylabel('Paper count'); plt.tight_layout(); plt.show()

## RAM và dung lượng đĩa

RSS cộng pipeline và child processes; shared pages có thể đếm nhiều lần. Peak là cực đại quan sát được. Disk là logical bytes trong output, không gồm results của sampler.

In [ ]:
display(pd.DataFrame([benchmark['ram']]))
display(pd.DataFrame([benchmark['disk']]))
frame = pd.DataFrame(samples)
if not frame.empty:
    fig, axes = plt.subplots(2,1,figsize=(9,6),sharex=True)
    axes[0].plot(frame['elapsed_seconds'],frame['rss_bytes']/1024**2); axes[0].set_ylabel('Summed RSS (MiB)')
    axes[1].plot(frame['elapsed_seconds'],frame['disk_bytes']/1024**2); axes[1].set_ylabel('Output size (MiB)'); axes[1].set_xlabel('Elapsed seconds')
    plt.tight_layout(); plt.show()